In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/time-series-financial-prediction/sample_submission.csv
/kaggle/input/competitions/time-series-financial-prediction/train.csv


In [6]:
#!pip install captum

In [7]:
!pip install "numpy>=2.0" "pandas==2.2.2" captum optuna

INFO: pip is looking at multiple versions of captum to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 94.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.2 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.3.3
    Uninstalling pandas-2.3.3:
      Successfully uninstalled pandas-2.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [8]:
# 1. Imports and Setup
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import optuna
from captum.attr import IntegratedGradients
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [9]:
# Load dataset
train_df = pd.read_csv('/kaggle/input/competitions/time-series-financial-prediction/train.csv')
company_ids = train_df['ID'].values
data_matrix = train_df.drop(columns=['ID']).values # Shape: (442, num_days)

SEQ_LEN = 30 # Use 30 days of history to predict the next day
VAL_DAYS = 100 # Reserve the last 100 days for validation

# Split into train and validation matrix
train_matrix = data_matrix[:, :-VAL_DAYS]
val_matrix = data_matrix[:, -(VAL_DAYS + SEQ_LEN):] # Need prior SEQ_LEN days for the first val prediction

def create_sequences(data, seq_len):
    """
    Transforms the 2D matrix (companies, days) into 3D sequences
    Returns: X (samples, seq_len, 1), y (samples)
    """
    xs, ys = [], []
    num_companies, num_days = data.shape
    for i in range(num_companies):
        for t in range(num_days - seq_len):
            xs.append(data[i, t : t+seq_len])
            ys.append(data[i, t+seq_len])
            
    # Reshape for PyTorch RNN input: (Batch, Sequence Length, Features)
    return np.array(xs)[..., np.newaxis], np.array(ys)

X_train, y_train = create_sequences(train_matrix, SEQ_LEN)
X_val, y_val = create_sequences(val_matrix, SEQ_LEN)

print(f"Train shapes - X: {X_train.shape}, y: {y_train.shape}")
print(f"Val shapes - X: {X_val.shape}, y: {y_val.shape}")

Train shapes - X: (1277822, 30, 1), y: (1277822,)
Val shapes - X: (44200, 30, 1), y: (44200,)


In [10]:
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Will initialize DataLoaders dynamically during Optuna tuning

In [11]:
class StockRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super(StockRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, features)
        out, _ = self.lstm(x)
        # Extract the outputs of the final time step
        out = out[:, -1, :] 
        out = self.fc(out)
        
        # Squeeze ONLY the last dimension so batch_size=1 doesn't become a 0-dim scalar
        return out.squeeze(-1)

In [12]:
def objective(trial):
    # Suggest hyperparameters
    hidden_size = trial.suggest_categorical('hidden_size', [32, 64, 128])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [128, 256, 512])
    
    # Loaders
    train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(StockDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    
    # Model Setup
    model = StockRNN(hidden_size=hidden_size, num_layers=num_layers).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Quick Training Loop for Trial
    epochs = 5 # Keep low for fast tuning, increase for full training
    for epoch in range(epochs):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            outputs = model(X_b)
            loss = criterion(outputs, y_b)
            loss.backward()
            optimizer.step()
            
    # Evaluation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            outputs = model(X_b)
            loss = criterion(outputs, y_b)
            val_loss += loss.item() * X_b.size(0)
            
    return val_loss / len(val_loader.dataset)

# Run Optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5) # Increase n_trials for better tuning!

print("\nBest trial:")
best_params = study.best_trial.params
print(best_params)

[I 2026-03-10 15:27:28,547] A new study created in memory with name: no-name-f9d74983-b405-46d9-8889-dd8c5050a66c
[I 2026-03-10 15:28:44,466] Trial 0 finished with value: 4.560905831841861 and parameters: {'hidden_size': 64, 'num_layers': 2, 'lr': 0.00021090512610017482, 'batch_size': 512}. Best is trial 0 with value: 4.560905831841861.
[I 2026-03-10 15:31:18,882] Trial 1 finished with value: 4.463892147357647 and parameters: {'hidden_size': 32, 'num_layers': 2, 'lr': 0.00823516378786067, 'batch_size': 128}. Best is trial 1 with value: 4.463892147357647.
[I 2026-03-10 15:32:22,869] Trial 2 finished with value: 4.6031365793323085 and parameters: {'hidden_size': 64, 'num_layers': 1, 'lr': 0.0005901081092250798, 'batch_size': 512}. Best is trial 1 with value: 4.463892147357647.
[I 2026-03-10 15:37:17,983] Trial 3 finished with value: 4.513286819889535 and parameters: {'hidden_size': 128, 'num_layers': 3, 'lr': 0.0014360917618828687, 'batch_size': 128}. Best is trial 1 with value: 4.463892


Best trial:
{'hidden_size': 32, 'num_layers': 2, 'lr': 0.00823516378786067, 'batch_size': 128}


In [13]:
# Retrain with Best Params on ALL Data
best_batch = best_params['batch_size']
X_full, y_full = create_sequences(data_matrix, SEQ_LEN)
full_loader = DataLoader(StockDataset(X_full, y_full), batch_size=best_batch, shuffle=True)

final_model = StockRNN(
    hidden_size=best_params['hidden_size'], 
    num_layers=best_params['num_layers']
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params['lr'])

# Final Training
epochs = 10 
final_model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for X_b, y_b in full_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        outputs = final_model(X_b)
        loss = criterion(outputs, y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss/len(full_loader):.4f}")

# Predict April 1st 2022
# Extract the last sequence (SEQ_LEN) for each company
last_sequences = data_matrix[:, -SEQ_LEN:]
last_sequences = last_sequences[..., np.newaxis] # (442, SEQ_LEN, 1)

final_model.eval()
with torch.no_grad():
    X_test = torch.tensor(last_sequences, dtype=torch.float32).to(device)
    predictions = final_model(X_test).cpu().numpy()


Epoch 1/10 - Loss: 3.6511
Epoch 2/10 - Loss: 3.5801
Epoch 3/10 - Loss: 3.5455
Epoch 4/10 - Loss: 3.5647
Epoch 5/10 - Loss: 3.6818
Epoch 6/10 - Loss: 3.6448
Epoch 7/10 - Loss: 3.6542
Epoch 8/10 - Loss: 3.6767
Epoch 9/10 - Loss: 3.6883
Epoch 10/10 - Loss: 3.6927


In [14]:
# Format Submission File
submission = pd.read_csv('/kaggle/input/competitions/time-series-financial-prediction/sample_submission.csv')
submission['value'] = predictions
submission.to_csv('submission.csv', index=False)
print("\nPredictions saved to submission.csv successfully!")


Predictions saved to submission.csv successfully!


In [16]:
# Interpret the first company's prediction
final_model.eval() # Keep in eval mode for deterministic behavior
company_0_seq = torch.tensor(last_sequences[0:1], dtype=torch.float32, requires_grad=True).to(device)

# Initialize Integrated Gradients
ig = IntegratedGradients(final_model)

# Calculate attributes
baseline = torch.zeros_like(company_0_seq).to(device)

# ---------------------------------------------------------
# WORKAROUND: Temporarily disable cuDNN to allow RNN backward 
# pass in eval mode, preventing the RuntimeError.
torch.backends.cudnn.enabled = False 
# ---------------------------------------------------------

attributions, delta = ig.attribute(company_0_seq, baseline, return_convergence_delta=True)

# Re-enable cuDNN for any future training/processing
torch.backends.cudnn.enabled = True 

attributions = attributions.squeeze().cpu().detach().numpy()

# Plot the attributions
plt.figure(figsize=(10, 4))
plt.bar(range(1, SEQ_LEN + 1), attributions, color='teal')
plt.title("Captum Integrated Gradients: Feature Attribution for Company 0")
plt.xlabel(f"Days in Sequence (1 to {SEQ_LEN}, where {SEQ_LEN} is the day before April 1st)")
plt.ylabel("Attribution Score")
plt.grid(alpha=0.3)
plt.show()

IndexError: tuple index out of range